# Gemini Function Calling: Dynamic Form Generation

**This notebook demonstrates how to use the Gemini API's function calling capabilities to dynamically generate and modify a JSON structure representing a web form.**

### Key Features:

*   **Natural Language to JSON:** Converts user prompts (e.g., "create a healthcare form") into a structured JSON object.
*   **Function Calling for Data Lookup:** Defines a `get_lookup_values` function that the Gemini model can call as a tool. This allows the form to be populated with dynamic data from an external source (a `lookups.csv` file).
*   **Contextual Modification:** The model maintains the context of the conversation, allowing users to progressively modify the form with simple follow-up commands (e.g., "Add a field for Signature").
*   **System Prompt Engineering:** A detailed system prompt guides the model's behavior, enforcing output format, tool usage, and task logic.

By the end of this notebook, you will see how to build a simple, interactive chat-based interface for creating complex, data-driven JSON objects.

Author - Navneet Tuteja

## 1. Setup and Installation

In [1]:
%pip install -qU 'google-genai>=1.0.0'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 8.7 MB/s eta 0:00:00


### Generate a lookups.csv file.

In [2]:
# code to create a csv file with headers lookup_name, lookup_values and write values to lookups.csv. Values are data_center_locations	USA-East

import csv

data = [
    {"lookup_name": "data_center_locations", "lookup_values": "USA-East"},
    {"lookup_name": "data_center_locations", "lookup_values": "USA-West"},
    {"lookup_name": "data_center_locations", "lookup_values": "EU-Central"},
    {"lookup_name": "data_center_locations", "lookup_values": "Asia-Pacific"},
    {"lookup_name": "issue_urgency", "lookup_values": "Low"},
    {"lookup_name": "issue_urgency", "lookup_values": "Medium"},
    {"lookup_name": "issue_urgency", "lookup_values": "High"},
    {"lookup_name": "issue_urgency", "lookup_values": "Critical"},
    {"lookup_name": "t_shirt_sizes", "lookup_values": "Small"},
    {"lookup_name": "t_shirt_sizes", "lookup_values": "Medium"},
]

with open('lookups.csv', 'w', newline='') as csvfile:
    fieldnames = ['lookup_name', 'lookup_values']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(data)

## 2. Import Libraries

In [3]:
# Import necessary libraries
import os  # For interacting with the operating system
import json  # For working with JSON data
import pandas as pd  # For data manipulation and analysis, especially with CSV files
from google import genai  # The Google Generative AI library
from google.genai import types  # Specific types from the genai library
import re  # For regular expressions
import base64  # For encoding and decoding data
from google.genai.types import FunctionDeclaration, GenerateContentConfig, Part, Tool  # Specific components for function calling

## 3. Configure Model and Tools

In [4]:
# Define the model ID for the Gemini API
# This specifies which version of the Gemini model will be used.
MODEL_ID="gemini-2.5-pro"

### Define a Tool for Lookup Values

In [5]:
from typing import Optional, List

# This function is defined as a tool that the Gemini model can call.
def get_lookup_values(field_name: Optional[str] = None) -> List[str]:
    """
    Looks up values from a local CSV file. If field_name is provided, it returns
    the corresponding lookup_values. If field_name is None, it returns all
    unique lookup_names.

    Args:
        field_name (Optional[str]): The lookup_name to search for. Defaults to None.

    Returns:
        List[str]: A list of matching values or a list of unique lookup names.
    """
    print(f"--- TOOLBOX: Performing lookup for: '{field_name}' ---")
    try:
        # Read the lookup data from a CSV file named 'lookups.csv'
        df = pd.read_csv("lookups.csv")
        # Check if a specific field_name is provided
        if field_name is None:
            # If no field_name is given, get all unique names from the 'lookup_name' column
            unique_names = df['lookup_name'].unique().tolist()
            print(f"✅ Found unique lookup names: {unique_names}")
            # Return the list of unique lookup names
            return unique_names
        else:
            # If a field_name is provided, filter the DataFrame to find rows where 'lookup_name' matches
            filtered_df = df[df['lookup_name'] == field_name]
            # Check if any matching rows were found
            if not filtered_df.empty:
                # If matches are found, get the values from the 'lookup_value' column
                # Drop any missing values (NaN), convert them to strings, and put them in a list
                values = filtered_df['lookup_value'].dropna().astype(str).tolist()
                print(f"✅ Found values for '{field_name}': {values}")
                # Return the list of lookup values
                return values
            else:
                # If no matching field_name is found, print a warning
                print(f"⚠️ Field '{field_name}' not found in lookups.csv.")
                return []
    # Handle the case where the 'lookups.csv' file does not exist
    except FileNotFoundError:
        print("🚨 ERROR: lookups.csv not found.")
        return []
    # Handle any other exceptions that might occur
    except Exception as e:
        print(f"🚨 ERROR in get_lookup_values: {e}")
        return []


# Example of calling the function to get all lookup names
# This demonstrates how to use the function without providing a field_name
get_lookup_values()

--- TOOLBOX: Performing lookup for: 'None' ---
✅ Found unique lookup names: ['data_center_locations', 'issue_urgency', 't_shirt_sizes']


['data_center_locations', 'issue_urgency', 't_shirt_sizes']

### Define the System Prompt

In [6]:

print("\n--- MANAGING GEMINI CHAT SESSION ---")

# --- Load lookup fields to inject into the prompt ---
# This section reads the available lookup fields from the CSV to inform the model.
try:
        # Load the lookup data from the CSV file into a pandas DataFrame.
        lookup_df = pd.read_csv("lookups.csv")
        # Get the unique lookup names to be used in the prompt.
        # This provides the model with a list of valid fields it can request from the tool.
        available_lookups =  lookup_df['lookup_name'].unique().tolist()
        # print(available_lookups) # Uncomment for debugging
except Exception:
        # If the file can't be read, provide a default message.
        available_lookups = "Not available. The lookup file could not be read."

print(f"ℹ️ Injecting available lookup fields into prompt: {available_lookups}")

# --- Define the System Prompt for the Gemini Model ---
# The system prompt guides the model's behavior, setting rules and providing context.
system_prompt = f"""
    You are a highly precise AI assistant proficient in creating forms as per user requests and modifying JSON for web forms. Since you are a specialist, you dont vcreate basic forms.

    **Primary Directive:** Generate a valid JSON object conforming to the schema.

    **Output Format Rules:**
    1.  Your response MUST START with a Markdown JSON code block: ```json ... ```.
    2.  Conversational text MUST come AFTER the JSON block.

    **Tool Usage for Lookups:**
    - If the user asks for a field implying predefined choices (e.g., "a dropdown for data centers"), you MUST use the `get_lookup_values` tool.
    - Here are the available lookup fields you can request in the `field_name` parameter of the tool: **[{available_lookups}]**
    - You must choose the best semantic match from that list for the `field_name` argument. For example, if the user says "urgency level", you should call the tool with `field_name='issue_urgency'`.
    - If the user provides their own list of options, use their list instead of the tool.

    **Task Logic:**
    - If "CURRENT_FORM_JSON" is not provided, create a new form.
    - If "CURRENT_FORM_JSON" is provided, modify it according to the new instruction.

    **Field Object Schema (Reminder):**
    `display_name`, `field_id`, `type`, `options` (for choice types), `required` (optional), `help_text` (optional).
    """





--- MANAGING GEMINI CHAT SESSION ---
ℹ️ Injecting available lookup fields into prompt: ['data_center_locations', 'issue_urgency', 't_shirt_sizes']


### Configure Generation Settings

In [7]:
# Configure the content generation settings for the Gemini model
generate_content_config = types.GenerateContentConfig(
    # Temperature controls the randomness of the output. Lower is more deterministic.
    temperature = .1,
    # The system_instruction provides the high-level guidance for the model's behavior.
    system_instruction=system_prompt,
    # The tools parameter makes the get_lookup_values function available to the model.
    tools= [get_lookup_values],
  )

## 4. Initialize and Start Chat

In [8]:
# Initialize the Generative AI client for Vertex AI
client = genai.Client(vertexai=True, project='learn-w-me', location='global')

# Create a new chat session with the specified model and configuration
# This object will maintain the conversation history.
chat = client.chats.create(
    # Specify the model to use for the chat
    model = MODEL_ID,
    # Apply the generation configuration defined in the previous cell
    config = generate_content_config
)

### Send Initial Message to Create a Form

In [9]:
# Send an initial message to the chat to create a healthcare form
# The model will use the system prompt and its training to generate a relevant form.
response = chat.send_message("""
  create a healthcare form
"""
)

# Print the 'text' part of the response from the model, which should contain the JSON form.
print(response.text)

```json
{
  "form_name": "Healthcare Form",
  "fields": [
    {
      "display_name": "Patient Name",
      "field_id": "patient_name",
      "type": "text",
      "required": true
    },
    {
      "display_name": "Date of Birth",
      "field_id": "date_of_birth",
      "type": "date",
      "required": true
    },
    {
      "display_name": "Gender",
      "field_id": "gender",
      "type": "radiogroup",
      "options": [
        "Male",
        "Female",
        "Other"
      ],
      "required": true
    },
    {
      "display_name": "Medical Record Number",
      "field_id": "medical_record_number",
      "type": "text",
      "required": true
    },
    {
      "display_name": "Reason for Visit",
      "field_id": "reason_for_visit",
      "type": "textarea"
    },
    {
      "display_name": "Insurance Provider",
      "field_id": "insurance_provider",
      "type": "text"
    },
    {
      "display_name": "Policy Number",
      "field_id": "policy_number",
      "type": 

### Review Chat History

In [10]:
# Print a header for the chat history section.
print("\n--- History ---")
# The `print_history` function (if defined) would typically iterate through
# the `chat.history` object to display the conversation turns.
# print_history(chat)


--- History ---


### Continue Chat - Modify the Existing Form

In [11]:
# Send another message to the chat to modify the existing form.
# The model will consider the previous conversation context (the healthcare form).
response = chat.send_message("""
  Add a field Signature
"""
)

# Print the updated form from the model's response.
print(response.text)

```json
{
  "form_name": "Healthcare Form",
  "fields": [
    {
      "display_name": "Patient Name",
      "field_id": "patient_name",
      "type": "text",
      "required": true
    },
    {
      "display_name": "Date of Birth",
      "field_id": "date_of_birth",
      "type": "date",
      "required": true
    },
    {
      "display_name": "Gender",
      "field_id": "gender",
      "type": "radiogroup",
      "options": [
        "Male",
        "Female",
        "Other"
      ],
      "required": true
    },
    {
      "display_name": "Medical Record Number",
      "field_id": "medical_record_number",
      "type": "text",
      "required": true
    },
    {
      "display_name": "Reason for Visit",
      "field_id": "reason_for_visit",
      "type": "textarea"
    },
    {
      "display_name": "Insurance Provider",
      "field_id": "insurance_provider",
      "type": "text"
    },
    {
      "display_name": "Policy Number",
      "field_id": "policy_number",
      "type": 